# KL-Descent Path — Distribution-Space Adaptive KL-IG

The path lives in **distribution space**, not pixel space.  A Gaussian
$N(\mu(s), \exp(lv(s))\cdot I)$ slides through $(\mu, lv)$-space by gradient
descent on the closed-form KL between two diagonal Gaussians:

$$\min_{\mu,lv}\;\text{KL}\!\left(N(\mu, e^{lv}\!I)\;\big\|\;N(\mu_{cf}, e^{lv_{cf}}\!I)\right)$$

Descent is purely geometric (no model autograd needed for the path itself).
After descent, the trajectory is reversed so $s\!=\!0$ is the counterfactual end
and $s\!=\!1$ is the explicand — matching KL-IG's baseline → explicand convention.
The existing `KLIntegratedGradients` integrator handles the rest unchanged.

**Three counterfactuals compared:**
1. **max-ent** — $\mu_{cf}\!=\!0,\;lv_{cf}\!=\!0$  (N(0,I), like LinearPath baseline)
2. **sharp 2nd** — $\mu_{cf}\!=$ another-class image, $lv_{cf}$ tight
3. **fuzzy 2nd** — $\mu_{cf}\!=$ another-class image, $lv_{cf}$ wide


In [ ]:
!git clone --branch claude/general-session-FcgoB \
    https://github.com/Shameen5375/KLIG_V1 /content/KLIG_V1 2>/dev/null || \
    git -C /content/KLIG_V1 fetch origin claude/general-session-FcgoB && \
    git -C /content/KLIG_V1 reset --hard origin/claude/general-session-FcgoB
!pip install -e /content/KLIG_V1/infocube-main -q
!pip install captum datasets tqdm scikit-learn -q

In [ ]:
import os, sys, math, json, pickle, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

for _root in ["/content/KLIG_V1/infocube-main", "infocube-main"]:
    if os.path.isdir(_root) and _root not in sys.path:
        sys.path.insert(0, _root)

from klig import KLIntegratedGradients, KLDescentPath
from klig.core.path import LinearPath
from torchvision.models import resnet50, ResNet50_Weights
print("imports OK")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
N_IMGS            = 100
N_STEPS           = 50      # integration quadrature points
N_SAMPLES         = 10      # MC samples per quadrature point
T_DESCENT         = 50      # max KL-descent steps
STEP_SIZE         = 0.05    # joint step in (μ, lv) units
KL_STOP           = 1e-3
SIGMA_FINAL       = 1 / 256
LV_CF_SHARP       = 2.0 * math.log(1 / 256)   # tight Gaussian around cf image
LV_CF_FUZZY       = 0.0                       # N(·, 1) — class neighborhood
LV_CF_MAXENT      = 0.0                       # N(0, 1) — explicit prior
N_INSERTION_STEPS = 50
FORCE_RECOMPUTE   = False
VIS_IMG_IDX       = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CACHE_DIR = Path("/content/drive/MyDrive/kldescent_cache")
except Exception:
    CACHE_DIR = Path("kldescent_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    "KL-IG (linear)",
    "KL-Descent (max-ent)",
    "KL-Descent (sharp 2nd)",
    "KL-Descent (fuzzy 2nd)",
]
COLORS = {
    "KL-IG (linear)":          "#333333",
    "KL-Descent (max-ent)":    "#4daf4a",
    "KL-Descent (sharp 2nd)":  "#e41a1c",
    "KL-Descent (fuzzy 2nd)":  "#377eb8",
}

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
weights    = ResNet50_Weights.IMAGENET1K_V2
model      = resnet50(weights=weights).to(DEVICE).eval()
preprocess = weights.transforms()
imagenet_labels = weights.meta["categories"]
print("ResNet50 loaded")

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def denormalize(x):
    return x.cpu() * _STD + _MEAN

def absmax_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    idx = a.abs().argmax(dim=0)
    return a.gather(0, idx.unsqueeze(0)).squeeze(0)

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
_cache_ds = CACHE_DIR / "dataset.pkl"
if not FORCE_RECOMPUTE and _cache_ds.exists():
    with open(_cache_ds, "rb") as f: dataset = pickle.load(f)
    print(f"[cache] dataset n={len(dataset)}")
else:
    from datasets import load_dataset as _hf
    _ds = _hf("evanarlian/imagenet_1k_resized_256", split="train", streaming=True)
    dataset = []
    for item in tqdm(_ds.take(N_IMGS * 4), desc="loading"):
        img = item["image"]
        if img.mode != "RGB": img = img.convert("RGB")
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(x)
            tgt  = int(logits.argmax(-1).item())
            conf = logits.softmax(-1)[0, tgt].item()
        if conf > 0.3:
            dataset.append({"x": x, "target": tgt, "idx": len(dataset)})
        if len(dataset) >= N_IMGS: break
    with open(_cache_ds, "wb") as f: pickle.dump(dataset, f)
    print(f"Collected {len(dataset)} images")

In [ ]:
# ── Counterfactual selection ─────────────────────────────────────────────────
# For each explicand, pick a different-class image as the contrastive cf.
# Random pairing keeps it cheap; we just need a non-trivial μ_cf.
rng = np.random.default_rng(0)
_targets = np.array([row["target"] for row in dataset])

def pick_cf_image(idx):
    """Return another image x_cf (1,C,H,W) from a different class."""
    cand = np.where(_targets != _targets[idx])[0]
    j = int(cand[rng.integers(len(cand))])
    return dataset[j]["x"]

for i in [0, 1, 2]:
    j_x = pick_cf_image(i)
    print(f"explicand {i} ({imagenet_labels[dataset[i]['target']]}) → cf class differs")

In [ ]:
# ── Attribution loop ──────────────────────────────────────────────────────────
_cache_attr = CACHE_DIR / "kldescent_attrs.pkl"

if not FORCE_RECOMPUTE and _cache_attr.exists():
    with open(_cache_attr, "rb") as f:
        all_attrs, all_path_meta = pickle.load(f)
    print("[cache] attrs loaded")
else:
    all_attrs     = {m: [] for m in METHODS}
    all_path_meta = []

    klig_baseline = KLIntegratedGradients(
        model, n_steps=N_STEPS, n_samples=N_SAMPLES,
        sigma_final=SIGMA_FINAL, device=DEVICE)

    for row in tqdm(dataset, desc="attributing"):
        x, tgt = row["x"], row["target"]
        x1 = x.squeeze(0).to(DEVICE)
        meta = {}

        # 1) KL-IG (linear) — fixed schedule baseline
        r0 = klig_baseline.attribute(x1, target=tgt)
        all_attrs["KL-IG (linear)"].append(absmax_collapse(r0.attr).cpu())

        # 2) KL-Descent (max-ent): μ_cf = 0,  lv_cf = 0   (N(0,I))
        path_me = KLDescentPath(
            mu_cf=torch.zeros_like(x1), lv_cf=LV_CF_MAXENT,
            T=T_DESCENT, step_size=STEP_SIZE, kl_stop=KL_STOP)
        ig_me = KLIntegratedGradients(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                       sigma_final=SIGMA_FINAL, path=path_me, device=DEVICE)
        r1 = ig_me.attribute(x1, target=tgt)
        all_attrs["KL-Descent (max-ent)"].append(absmax_collapse(r1.attr).cpu())
        meta["maxent_kl"]  = list(path_me.kl_trajectory)
        meta["maxent_len"] = path_me.path_length

        # 3) KL-Descent (sharp 2nd): μ_cf = other-class image, lv_cf tight
        x_cf = pick_cf_image(row["idx"]).squeeze(0).to(DEVICE)
        path_sharp = KLDescentPath(
            mu_cf=x_cf, lv_cf=LV_CF_SHARP,
            T=T_DESCENT, step_size=STEP_SIZE, kl_stop=KL_STOP)
        ig_sharp = KLIntegratedGradients(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                          sigma_final=SIGMA_FINAL, path=path_sharp, device=DEVICE)
        r2 = ig_sharp.attribute(x1, target=tgt)
        all_attrs["KL-Descent (sharp 2nd)"].append(absmax_collapse(r2.attr).cpu())
        meta["sharp_kl"]  = list(path_sharp.kl_trajectory)
        meta["sharp_len"] = path_sharp.path_length

        # 4) KL-Descent (fuzzy 2nd): μ_cf = other-class image, lv_cf wide
        path_fuzzy = KLDescentPath(
            mu_cf=x_cf, lv_cf=LV_CF_FUZZY,
            T=T_DESCENT, step_size=STEP_SIZE, kl_stop=KL_STOP)
        ig_fuzzy = KLIntegratedGradients(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                          sigma_final=SIGMA_FINAL, path=path_fuzzy, device=DEVICE)
        r3 = ig_fuzzy.attribute(x1, target=tgt)
        all_attrs["KL-Descent (fuzzy 2nd)"].append(absmax_collapse(r3.attr).cpu())
        meta["fuzzy_kl"]  = list(path_fuzzy.kl_trajectory)
        meta["fuzzy_len"] = path_fuzzy.path_length

        all_path_meta.append(meta)

    with open(_cache_attr, "wb") as f:
        pickle.dump((all_attrs, all_path_meta), f)
    print("Done.")

In [ ]:
# ── 1. Path sanity: KL trajectories ───────────────────────────────────────────
N_SHOW = 8
fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor="white")
for i in range(min(N_SHOW, len(all_path_meta))):
    axes[0].plot(all_path_meta[i]["maxent_kl"], alpha=0.6, lw=1)
    axes[1].plot(all_path_meta[i]["sharp_kl"],  alpha=0.6, lw=1)
    axes[2].plot(all_path_meta[i]["fuzzy_kl"],  alpha=0.6, lw=1)
for ax, title in zip(axes, ["max-ent", "sharp 2nd", "fuzzy 2nd"]):
    ax.set_xlabel("Descent step"); ax.set_ylabel("KL")
    ax.set_title(f"KL-Descent — {title}")
    ax.axhline(KL_STOP, color="red", lw=1, ls="--", label=f"kl_stop={KL_STOP}")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle("Path sanity: KL should decrease monotonically", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

lens = {k: [m[f"{k}_len"] for m in all_path_meta] for k in ["maxent", "sharp", "fuzzy"]}
fig, ax = plt.subplots(figsize=(7, 3), facecolor="white")
for k, c in zip(["maxent", "sharp", "fuzzy"], ["#4daf4a", "#e41a1c", "#377eb8"]):
    ax.hist(lens[k], bins=20, alpha=0.5, label=k, color=c)
ax.axvline(T_DESCENT + 1, color="black", lw=1, ls="--", label=f"T+1={T_DESCENT+1}")
ax.set_xlabel("Path length"); ax.set_ylabel("Count")
ax.set_title("Path length distribution"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
for k in ["maxent", "sharp", "fuzzy"]:
    print(f"{k:8s}: mean={np.mean(lens[k]):.1f}  max={max(lens[k])}")

In [ ]:
# ── 2. Visualize μ(s) as image along the path ────────────────────────────────
# Rebuild one path explicitly so we can sample μ at several s.
row_v = dataset[VIS_IMG_IDX]
x1    = row_v["x"].squeeze(0).to(DEVICE)
x_cf  = pick_cf_image(VIS_IMG_IDX).squeeze(0).to(DEVICE)

path_viz = KLDescentPath(mu_cf=x_cf, lv_cf=LV_CF_SHARP,
                          T=T_DESCENT, step_size=STEP_SIZE, kl_stop=KL_STOP)
# trigger build
_ = path_viz.at(0.5, x1, torch.full_like(x1, 2.0 * math.log(SIGMA_FINAL)))

s_values = np.linspace(0.0, 1.0, 7)
fig, axes = plt.subplots(1, len(s_values), figsize=(2.5 * len(s_values), 2.7), facecolor="white")
for ax, s in zip(axes, s_values):
    mu_s, _ = path_viz.at(float(s), x1, torch.full_like(x1, 2.0 * math.log(SIGMA_FINAL)))
    img = np.clip(denormalize(mu_s).permute(1, 2, 0).numpy(), 0, 1)
    ax.imshow(img); ax.axis("off"); ax.set_title(f"s={s:.2f}", fontsize=9)
plt.suptitle(f"μ(s) along KL-Descent path  (cf=sharp other-class)",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()
print("s=0 should look like the counterfactual; s=1 should look like the explicand.")

In [ ]:
# ── 3. Attribution maps (single image) ────────────────────────────────────────
img_np = np.clip(denormalize(row_v["x"][0]).permute(1, 2, 0).numpy(), 0, 1)

fig, axes = plt.subplots(1, 1 + len(METHODS),
                          figsize=(3.0 * (1 + len(METHODS)), 3.2), facecolor="white")
axes[0].imshow(img_np); axes[0].axis("off")
axes[0].set_title("Original", fontsize=10, fontweight="bold")
for ax, m in zip(axes[1:], METHODS):
    a    = all_attrs[m][VIS_IMG_IDX].numpy()
    vmax = max(float(np.percentile(np.abs(a), 99)), 1e-9)
    ax.imshow(a, cmap="RdBu_r", vmin=-vmax, vmax=vmax); ax.axis("off")
    ax.set_title(m, fontsize=8, fontweight="bold", color=COLORS[m])
plt.suptitle(f"Attribution maps — {imagenet_labels[row_v['target']]}",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ── 4. Insertion / Deletion AUC ──────────────────────────────────────────────
_cache_id = CACHE_DIR / "kldescent_ins_del.pkl"

def insertion_deletion(model, x, attr_map, target, n_steps=N_INSERTION_STEPS):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]
    order = attr_map.detach().view(-1).abs().argsort(descending=True)
    pps   = max(1, H * W // n_steps)
    blur  = F.avg_pool2d(x, 31, 1, 15)
    x_ins, x_del = blur.clone(), x.clone()
    ins_s, del_s = [], []
    with torch.no_grad():
        for step in range(n_steps):
            pix = order[step * pps:(step + 1) * pps]
            for ch in range(C):
                x_ins[:, ch].reshape(-1)[pix] = x[:, ch].reshape(-1)[pix]
                x_del[:, ch].reshape(-1)[pix] = blur[:, ch].reshape(-1)[pix]
            ins_s.append(model(x_ins).softmax(-1)[0, target].item())
            del_s.append(model(x_del).softmax(-1)[0, target].item())
    return float(np.trapz(ins_s) / n_steps), float(np.trapz(del_s) / n_steps)

if not FORCE_RECOMPUTE and _cache_id.exists():
    with open(_cache_id, "rb") as f: ins_auc, del_auc = pickle.load(f)
else:
    ins_auc = defaultdict(list); del_auc = defaultdict(list)
    for row in tqdm(dataset[:N_IMGS], desc="ins/del"):
        x, tgt = row["x"], row["target"]
        for m in METHODS:
            attr = all_attrs[m][row["idx"]].to(DEVICE).unsqueeze(0)
            i, d = insertion_deletion(model, x, attr, tgt)
            ins_auc[m].append(i); del_auc[m].append(d)
    ins_auc, del_auc = dict(ins_auc), dict(del_auc)
    with open(_cache_id, "wb") as f: pickle.dump((ins_auc, del_auc), f)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor="white")
for ax, (title, aucs) in zip(axes, [("Insertion AUC ↑", ins_auc), ("Deletion AUC ↓", del_auc)]):
    for xi, m in enumerate(METHODS):
        v = aucs[m]; mu = np.mean(v); ci = 1.96 * np.std(v) / len(v) ** 0.5
        ax.bar(xi, mu, color=COLORS[m], alpha=0.85, width=0.6)
        ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=20, ha="right", fontsize=9)
    ax.set_title(title)
plt.suptitle(f"Insertion / Deletion AUC  (n={N_IMGS})", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5. Sensitivity-n (signed) ─────────────────────────────────────────────────
_cache_sn = CACHE_DIR / "kldescent_sens_n.pkl"

def sensitivity_n(model, x, attr_map, target, n_subsets=50, subset_size=0.1):
    rng = np.random.default_rng(42)
    attr_flat = attr_map.cpu().detach().view(-1).numpy()
    n_pix = attr_flat.size
    n_sel = max(1, int(n_pix * subset_size))
    corrs = []
    with torch.no_grad():
        f_x = model(x).softmax(-1)[0, target].item()
        for _ in range(n_subsets):
            idx    = rng.choice(n_pix, n_sel, replace=False)
            x_mask = x.clone()
            for ch in range(x.shape[1]):
                x_mask[:, ch].reshape(-1)[idx] = 0
            f_mask  = model(x_mask).softmax(-1)[0, target].item()
            delta_f = f_x - f_mask
            delta_a = float(attr_flat[idx].sum())   # signed
            corrs.append((delta_f, delta_a))
    df = np.array([c[0] for c in corrs])
    da = np.array([c[1] for c in corrs])
    if df.std() < 1e-9 or da.std() < 1e-9: return 0.0
    return float(np.corrcoef(df, da)[0, 1])

if not FORCE_RECOMPUTE and _cache_sn.exists():
    with open(_cache_sn, "rb") as f: sens_n = pickle.load(f)
else:
    sens_n = defaultdict(list)
    for row in tqdm(dataset, desc="sensitivity-n"):
        x, tgt = row["x"], row["target"]
        for m in METHODS:
            attr = all_attrs[m][row["idx"]].to(DEVICE).unsqueeze(0)
            sens_n[m].append(sensitivity_n(model, x, attr, tgt))
    sens_n = dict(sens_n)
    with open(_cache_sn, "wb") as f: pickle.dump(sens_n, f)

fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
for xi, m in enumerate(METHODS):
    v = sens_n[m]; mu = np.mean(v); ci = 1.96 * np.std(v) / len(v) ** 0.5
    ax.bar(xi, mu, color=COLORS[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
ax.set_xticks(range(len(METHODS)))
ax.set_xticklabels(METHODS, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Pearson r (signed)"); ax.set_title(f"Sensitivity-n  (n={len(dataset)})")
plt.tight_layout(); plt.show()

In [ ]:
# ── 6. Sparsity (Gini) ────────────────────────────────────────────────────────
def gini(v):
    v = v.abs().flatten().numpy()
    v = np.sort(v); n = len(v)
    return float((2 * np.arange(1, n + 1) - n - 1) @ v / (n * v.sum() + 1e-12))

gini_scores = {m: [gini(all_attrs[m][i]) for i in range(len(dataset))] for m in METHODS}

fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
for xi, m in enumerate(METHODS):
    v = gini_scores[m]; mu = np.mean(v); ci = 1.96 * np.std(v) / len(v) ** 0.5
    ax.bar(xi, mu, color=COLORS[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
ax.set_xticks(range(len(METHODS)))
ax.set_xticklabels(METHODS, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Gini ↑"); ax.set_title(f"Sparsity  (n={len(dataset)})")
plt.tight_layout(); plt.show()

In [ ]:
# ── 7. Completeness check ────────────────────────────────────────────────────
# KL-IG completeness: Σ attr ≈ E[f(x_endpoint)] − E[f(x_baseline)]
# baseline of KL-Descent path is the counterfactual end of the trajectory.
N_CC      = 20
N_PRIOR_M = 50

def expected_f(x, target, n=N_PRIOR_M, lv=0.0):
    """E_{z ~ N(x, e^lv I)}[ softmax(model(z))[target] ]."""
    with torch.no_grad():
        std = math.exp(0.5 * lv)
        z = x.unsqueeze(0) + std * torch.randn(n, *x.shape, device=x.device)
        return float(model(z).softmax(-1)[:, target].mean().item())

rows_cc = []
for i, row in enumerate(tqdm(dataset[:N_CC], desc="completeness")):
    x, tgt = row["x"], row["target"]
    x1     = x.squeeze(0).to(DEVICE)
    x_cf   = pick_cf_image(row["idx"]).squeeze(0).to(DEVICE)
    entry  = {}

    # f at explicand (tight Gaussian)
    f_expl_sharp = expected_f(x1,   tgt, lv=LV_CF_SHARP)
    f_expl_maxe  = expected_f(x1,   tgt, lv=LV_CF_MAXENT)
    f_expl_fuzzy = expected_f(x1,   tgt, lv=LV_CF_FUZZY)

    f_me  = expected_f(torch.zeros_like(x1), tgt, lv=LV_CF_MAXENT)
    f_sh  = expected_f(x_cf,                  tgt, lv=LV_CF_SHARP)
    f_fz  = expected_f(x_cf,                  tgt, lv=LV_CF_FUZZY)

    entry["delta_me"] = f_expl_maxe  - f_me
    entry["delta_sh"] = f_expl_sharp - f_sh
    entry["delta_fz"] = f_expl_fuzzy - f_fz
    entry["sum_me"]   = float(all_attrs["KL-Descent (max-ent)"][i].sum())
    entry["sum_sh"]   = float(all_attrs["KL-Descent (sharp 2nd)"][i].sum())
    entry["sum_fz"]   = float(all_attrs["KL-Descent (fuzzy 2nd)"][i].sum())
    rows_cc.append(entry)

print(f"\nCompleteness  (n={N_CC})")
print(f"{'cf':<10} {'mean|err|':>10} {'rel%':>7} {'pass<5%':>8}")
print("-" * 40)
for tag in ["me", "sh", "fz"]:
    deltas = np.array([r[f"delta_{tag}"] for r in rows_cc])
    sums   = np.array([r[f"sum_{tag}"]   for r in rows_cc])
    err    = np.abs(sums - deltas)
    rel    = err / (np.abs(deltas) + 1e-8) * 100
    passed = (rel < 5.0).mean() * 100
    print(f"{tag:<10} {err.mean():>10.4f} {rel.mean():>6.2f}%  {passed:>7.1f}%")

In [ ]:
# ── 8. Summary table ──────────────────────────────────────────────────────────
import pandas as pd

ci95 = lambda v: 1.96 * np.std(v) / len(v) ** 0.5
rows_sum = []
for m in METHODS:
    row_s = {"Method": m}
    if gini_scores.get(m):
        v = gini_scores[m]; row_s["Gini ↑"] = f"{np.mean(v):.3f}±{ci95(v):.3f}"
    if ins_auc.get(m):
        v = ins_auc[m];     row_s["Ins AUC ↑"] = f"{np.mean(v):.3f}±{ci95(v):.3f}"
    if del_auc.get(m):
        v = del_auc[m];     row_s["Del AUC ↓"] = f"{np.mean(v):.3f}±{ci95(v):.3f}"
    if sens_n.get(m):
        v = sens_n[m];      row_s["Sens-n ↑"] = f"{np.mean(v):.3f}±{ci95(v):.3f}"
    rows_sum.append(row_s)

df_sum = pd.DataFrame(rows_sum).set_index("Method")
print(df_sum.to_string())
df_sum